## NLP Final
### Part 4: ABSA
### Aren Mizuno
### March 12, 2026

In [61]:
# Imports
!pip -q install pyarrow transformers tqdm

import os
import ast
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive
from IPython.display import display
from textwrap import shorten

In [62]:
# Mount drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [92]:
# Paths
NER_PATH = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/ner.parquet"
BERTOPIC_PATH = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_text.parquet"
MODEL_DIR = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/sentiment_model"

OUTPUT_DIR = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

GRAPH_DIR = os.path.join(OUTPUT_DIR, "graphs")
TOPIC_GRAPH_DIR = os.path.join(GRAPH_DIR, "topic_article_sentiment")
COMPANY_GRAPH_DIR = os.path.join(GRAPH_DIR, "topic_top10_companies_sentiment")
TECH_GRAPH_DIR = os.path.join(GRAPH_DIR, "topic_top10_technologies_sentiment")
TIME_GRAPH_DIR = os.path.join(GRAPH_DIR, "topic_sentiment_over_time")

os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(TOPIC_GRAPH_DIR, exist_ok=True)
os.makedirs(COMPANY_GRAPH_DIR, exist_ok=True)
os.makedirs(TECH_GRAPH_DIR, exist_ok=True)
os.makedirs(TIME_GRAPH_DIR, exist_ok=True)

In [66]:
# Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Loaded from:", MODEL_DIR)
print("Device:", device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded from: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/sentiment_model
Device: cuda


In [67]:
# Load data ner
ner = pd.read_parquet(NER_PATH)
print("ner shape:", ner.shape)
display(ner.head(2))

ner shape: (136233, 8)


,url,date,title,text,text_for_extraction,text_clean,company,technology
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","bad idea ai price (bad), market cap, price tod...",[Bad Idea AI Price],[]
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,"[Instagram, Trump, Werners AI Art]",[]


In [68]:
# Load data bertopic
bertopic_text = pd.read_parquet(BERTOPIC_PATH, engine="pyarrow")
print("bertopic_text shape:", bertopic_text.shape)
display(bertopic_text.head(2))

bertopic_text shape: (136233, 5)


,url,date,title,text,bertopic_topic_text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",bad idea ai price bad market cap price today c...,10
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,0


In [69]:
# Topic label matching
topic_map = {
    -1: "Cross-Industry AI News / Outliers",
    0: "Enterprise Technology",
    1: "Consumer & Public Services",
    2: "Financial Services & Business",
    3: "Corporate AI Announcements",
    4: "Enterprise Software & Cybersecurity",
    5: "Healthcare & Life Sciences",
    6: "Financial Markets & Investing",
    7: "Media, Journalism & Public Policy",
    8: "Business Operations & Enterprise Automation",
    9: "Workforce & Compensation",
    10: "Cryptocurrency & Digital Assets",
    11: "Publishing & Intellectual Property",
    12: "Public AI Companies",
    13: "Technology Industry News",
    14: "Music & Entertainment",
    15: "African Technology Development",
    16: "AI Market Commentary"
}

In [70]:
# Helper function
def normalize_text(s):
    if s is None:
        return ""
    if isinstance(s, float) and pd.isna(s):
        return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def get_context(text, aspect, window_chars=400, max_total_chars=700):
    """
    If aspect appears in text, grab local context around it.
    Otherwise use the first chunk of the article.
    """
    text = normalize_text(text)
    aspect = normalize_text(aspect)

    if not text:
        return ""

    low_text = text.lower()
    low_aspect = aspect.lower()

    idx = low_text.find(low_aspect)
    if idx == -1:
        return text[:max_total_chars]

    start = max(0, idx - window_chars)
    end = min(len(text), idx + len(aspect) + window_chars)
    snippet = text[start:end]

    if len(snippet) > max_total_chars:
        snippet = snippet[:max_total_chars]

    return snippet

def safe_mode(series):
    vc = series.value_counts(dropna=True)
    return vc.index[0] if len(vc) > 0 else np.nan

def slugify(text, max_len=120):
    text = str(text)
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    text = re.sub(r"_+", "_", text)
    if len(text) > max_len:
        text = text[:max_len]
    return text if text else "untitled"

def wrap_or_shorten_labels(labels, max_len=35):
    return [shorten(str(x), width=max_len, placeholder="...") for x in labels]

def ensure_sentiment_columns(pivot_df, sentiments):
    for s in sentiments:
        if s not in pivot_df.columns:
            pivot_df[s] = 0
    return pivot_df[sentiments]

In [71]:
# Keep only needed columns
ner_keep = ["url", "text", "title", "company", "technology", "date"]

bertopic_keep = ["url", "bertopic_topic_text"]

ner_small = ner[ner_keep].copy()
bertopic_small = bertopic_text[bertopic_keep].copy()

ner_small = ner_small.drop_duplicates(subset=["url"])
bertopic_small = bertopic_small.drop_duplicates(subset=["url"])


In [72]:
# Map labels
bertopic_small["bertopic_topic_label"] = bertopic_small["bertopic_topic_text"].map(topic_map)

In [73]:
# Merge on url
df = ner_small.merge(bertopic_small, on="url", how="inner", suffixes=("_ner", "_bertopic"))

print("Merged shape:", df.shape)
display(df.head(2))

Merged shape: (136233, 8)


,url,text,title,company,technology,date,bertopic_topic_text,bertopic_topic_label
0,https://blockworks.co/price/bad,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",[Bad Idea AI Price],[],2025-06-23,10,Cryptocurrency & Digital Assets
1,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,"[Instagram, Trump, Werners AI Art]",[],2024-07-01,0,Enterprise Technology


In [74]:
# Build aspect row
rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Building aspect rows"):
    url = row["url"]
    title = row["title"]
    full_text = row["text"]
    topic_num = row["bertopic_topic_text"]
    topic_label = row["bertopic_topic_label"]
    date_val = row["date"]

    for aspect in row["company"]:
        context = get_context(full_text, aspect, window_chars=400, max_total_chars=700)
        rows.append({
            "url": url,
            "title": title,
            "date": date_val,
            "bertopic_topic_num": topic_num,
            "bertopic_topic_label": topic_label,
            "aspect_type": "company",
            "aspect": aspect,
            "context": context
        })

    for aspect in row["technology"]:
        context = get_context(full_text, aspect, window_chars=400, max_total_chars=700)
        rows.append({
            "url": url,
            "title": title,
            "date": date_val,
            "bertopic_topic_num": topic_num,
            "bertopic_topic_label": topic_label,
            "aspect_type": "technology",
            "aspect": aspect,
            "context": context
        })

aspect_df = pd.DataFrame(rows)

aspect_df = aspect_df[
    aspect_df["aspect"].astype(str).str.strip().ne("") &
    aspect_df["context"].astype(str).str.strip().ne("")
].copy()

print("Aspect rows shape:", aspect_df.shape)
display(aspect_df.head(10))

# Optional
TOP_N_COMMON = None  # set to int if needed

if TOP_N_COMMON is not None:
    top_aspects = (
        aspect_df.groupby(["aspect_type", "aspect"])
        .size()
        .reset_index(name="mention_count")
        .sort_values(["aspect_type", "mention_count"], ascending=[True, False])
        .groupby("aspect_type", group_keys=False)
        .head(TOP_N_COMMON)
    )

    aspect_df = aspect_df.merge(
        top_aspects[["aspect_type", "aspect"]],
        on=["aspect_type", "aspect"],
        how="inner"
    ).copy()

    print("Aspect rows after top-N filter:", aspect_df.shape)

Building aspect rows:   0%|          | 0/136233 [00:00<?, ?it/s]

Aspect rows shape: (601313, 8)


,url,title,date,bertopic_topic_num,bertopic_topic_label,aspect_type,aspect,context
0,https://blockworks.co/price/bad,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",2025-06-23,10,Cryptocurrency & Digital Assets,company,Bad Idea AI Price,"Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Instagram,hot: Luma AI I'm sure by now you're tired of l...
2,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Trump,s AI video attempt to show gymnastics is one o...
3,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Werners AI Art,"m Mon Jul 1, 2024 Screenshot: Luma AI I'm sure..."
4,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Apple,erfectly functional format of Diskprices. Web ...
5,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Lifetime Subscription,erful Products (Contact Support) Newsletter : ...
6,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Frenchie,et Stand is on sale for just $139.99 (reg. $15...
7,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,technology,ai model,e trying to use it. If you want to make a blog...
8,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,technology,fine-tuning,more jumping from one app to another just to p...
9,https://citylife.capetown/gl/uncategorized/the...,The Road Ahead: How China's AI Foundation Mode...,2023-11-10,0,Enterprise Technology,company,Baidu,"s, China has emerged as a global leader in the..."


In [75]:
# Label mapping
id2label = getattr(model.config, "id2label", None)

if (not id2label) or all(str(v).startswith("LABEL_") for v in id2label.values()):
    id2label = {
        0: "negative",
        1: "neutral",
        2: "positive"
    }

print("Label mapping:", id2label)

label_names_lower = {str(v).lower() for v in id2label.values()}
can_make_score = {"negative", "neutral", "positive"}.issubset(label_names_lower)

Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [76]:
# Predict ABSA
def predict_absa(df_in, tokenizer, model, device, batch_size=32, max_length=256):
    all_pred_ids = []
    all_pred_labels = []
    all_probs = []

    contexts = df_in["context"].tolist()
    aspects = df_in["aspect"].tolist()

    for i in tqdm(range(0, len(df_in), batch_size), desc="Running ABSA"):
        batch_contexts = contexts[i:i+batch_size]
        batch_aspects = aspects[i:i+batch_size]

        enc = tokenizer(
            batch_contexts,
            text_pair=batch_aspects,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
            pred_ids = probs.argmax(axis=1)

        for pid, pvec in zip(pred_ids, probs):
            all_pred_ids.append(int(pid))
            all_pred_labels.append(str(id2label[int(pid)]).lower())
            all_probs.append(pvec)

    out = df_in.copy()
    out["sentiment_id"] = all_pred_ids
    out["sentiment"] = all_pred_labels

    probs_arr = np.vstack(all_probs)
    for class_id in range(probs_arr.shape[1]):
        label_name = str(id2label[class_id]).lower()
        out[f"prob_{label_name}"] = probs_arr[:, class_id]

    if can_make_score:
        score_map = {"negative": -1, "neutral": 0, "positive": 1}
        out["sentiment_score"] = out["sentiment"].map(score_map)
    else:
        out["sentiment_score"] = np.nan

    return out

absa_df = predict_absa(
    df_in=aspect_df,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=256
)

print("ABSA output shape:", absa_df.shape)
display(absa_df.head(10))

Running ABSA:   0%|          | 0/18792 [00:00<?, ?it/s]

ABSA output shape: (601313, 14)


,url,title,date,bertopic_topic_num,bertopic_topic_label,aspect_type,aspect,context,sentiment_id,sentiment,prob_negative,prob_neutral,prob_positive,sentiment_score
0,https://blockworks.co/price/bad,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",2025-06-23,10,Cryptocurrency & Digital Assets,company,Bad Idea AI Price,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",1,neutral,0.023605,0.973091,0.003304,0
1,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Instagram,hot: Luma AI I'm sure by now you're tired of l...,1,neutral,0.027618,0.960470,0.011913,0
2,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Trump,s AI video attempt to show gymnastics is one o...,1,neutral,0.480598,0.502365,0.017037,0
3,https://boingboing.net/2024/07/01/this-ai-vide...,This AI video of gymnastics might be the freak...,2024-07-01,0,Enterprise Technology,company,Werners AI Art,"m Mon Jul 1, 2024 Screenshot: Luma AI I'm sure...",1,neutral,0.028334,0.961605,0.010062,0
4,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Apple,erfectly functional format of Diskprices. Web ...,1,neutral,0.005597,0.886996,0.107407,0
5,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Lifetime Subscription,erful Products (Contact Support) Newsletter : ...,1,neutral,0.002817,0.989016,0.008166,0
6,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,company,Frenchie,et Stand is on sale for just $139.99 (reg. $15...,1,neutral,0.006160,0.973705,0.020135,0
7,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,technology,ai model,e trying to use it. If you want to make a blog...,1,neutral,0.003533,0.991400,0.005067,0
8,https://boingboing.net/2024/09/18/if-using-ai-...,"If using AI feels like a chore, try this - Boi...",2024-09-22,0,Enterprise Technology,technology,fine-tuning,more jumping from one app to another just to p...,1,neutral,0.002193,0.987968,0.009839,0
9,https://citylife.capetown/gl/uncategorized/the...,The Road Ahead: How China's AI Foundation Mode...,2023-11-10,0,Enterprise Technology,company,Baidu,"s, China has emerged as a global leader in the...",2,positive,0.001893,0.089839,0.908268,1


In [79]:
# Aggregate by aspect (company/technology)
prob_cols = [c for c in absa_df.columns if c.startswith("prob_")]

agg_dict = {
    "url": "nunique",
    "aspect": "size"
}
for c in prob_cols:
    agg_dict[c] = "mean"
if "sentiment_score" in absa_df.columns:
    agg_dict["sentiment_score"] = "mean"

# Main aspect-level aggregate: no topic grouping
aspect_agg_df = (
    absa_df.groupby(["aspect_type", "aspect"])
    .agg(agg_dict)
    .rename(columns={
        "url": "n_unique_articles",
        "aspect": "n_mentions"
    })
    .reset_index()
)

# Dominant sentiment for each aspect
dominant_aspect_sentiment = (
    absa_df.groupby(["aspect_type", "aspect"])["sentiment"]
    .agg(safe_mode)
    .reset_index(name="dominant_sentiment")
)

# Dominant topic for each aspect
dominant_aspect_topic = (
    absa_df.groupby(["aspect_type", "aspect"])["bertopic_topic_label"]
    .agg(safe_mode)
    .reset_index(name="dominant_topic")
)

aspect_agg_df = aspect_agg_df.merge(
    dominant_aspect_sentiment,
    on=["aspect_type", "aspect"],
    how="left"
)

aspect_agg_df = aspect_agg_df.merge(
    dominant_aspect_topic,
    on=["aspect_type", "aspect"],
    how="left"
)

aspect_agg_df = aspect_agg_df.sort_values(
    ["aspect_type", "n_mentions"],
    ascending=[True, False]
).reset_index(drop=True)

# Split into separate company / technology tables
company_aspect_agg_df = (
    aspect_agg_df[aspect_agg_df["aspect_type"] == "company"]
    .copy()
    .reset_index(drop=True)
)

technology_aspect_agg_df = (
    aspect_agg_df[aspect_agg_df["aspect_type"] == "technology"]
    .copy()
    .reset_index(drop=True)
)

print("Aspect aggregate shape:", aspect_agg_df.shape)
display(aspect_agg_df.head(20))

print("\nCompany aspect aggregate shape:", company_aspect_agg_df.shape)
display(company_aspect_agg_df.head(20))

print("\nTechnology aspect aggregate shape:", technology_aspect_agg_df.shape)
display(technology_aspect_agg_df.head(20))

Aspect aggregate shape: (71910, 10)


,aspect_type,aspect,n_unique_articles,n_mentions,prob_negative,prob_neutral,prob_positive,sentiment_score,dominant_sentiment,dominant_topic
0,company,Google,21352,21352,0.139655,0.668384,0.191961,0.028803,neutral,Enterprise Technology
1,company,Microsoft,15467,15467,0.179129,0.590218,0.230654,0.033297,neutral,Enterprise Technology
2,company,Amazon,10361,10361,0.158443,0.587168,0.254390,0.083776,neutral,Enterprise Technology
3,company,Apple,8024,8024,0.171176,0.608252,0.220573,0.030907,neutral,Enterprise Technology
4,company,Instagram,6570,6570,0.143123,0.728548,0.128329,-0.030746,neutral,Enterprise Technology
5,company,Meta,6280,6280,0.194654,0.603368,0.201978,-0.011465,neutral,Enterprise Technology
6,company,Nvidia,5139,5139,0.179948,0.447390,0.372662,0.188947,neutral,Enterprise Technology
7,company,Facebook,4954,4954,0.088585,0.825496,0.085920,-0.009891,neutral,Enterprise Technology
8,company,Nexstar Media Inc,2760,2760,0.096273,0.852398,0.051328,-0.052899,neutral,Consumer & Public Services
9,company,Tesla,2530,2530,0.257936,0.572220,0.169844,-0.109881,neutral,Enterprise Technology



Company aspect aggregate shape: (71885, 10)


,aspect_type,aspect,n_unique_articles,n_mentions,prob_negative,prob_neutral,prob_positive,sentiment_score,dominant_sentiment,dominant_topic
0,company,Google,21352,21352,0.139655,0.668384,0.191961,0.028803,neutral,Enterprise Technology
1,company,Microsoft,15467,15467,0.179129,0.590218,0.230654,0.033297,neutral,Enterprise Technology
2,company,Amazon,10361,10361,0.158443,0.587168,0.254390,0.083776,neutral,Enterprise Technology
3,company,Apple,8024,8024,0.171176,0.608252,0.220573,0.030907,neutral,Enterprise Technology
4,company,Instagram,6570,6570,0.143123,0.728548,0.128329,-0.030746,neutral,Enterprise Technology
5,company,Meta,6280,6280,0.194654,0.603368,0.201978,-0.011465,neutral,Enterprise Technology
6,company,Nvidia,5139,5139,0.179948,0.447390,0.372662,0.188947,neutral,Enterprise Technology
7,company,Facebook,4954,4954,0.088585,0.825496,0.085920,-0.009891,neutral,Enterprise Technology
8,company,Nexstar Media Inc,2760,2760,0.096273,0.852398,0.051328,-0.052899,neutral,Consumer & Public Services
9,company,Tesla,2530,2530,0.257936,0.572220,0.169844,-0.109881,neutral,Enterprise Technology



Technology aspect aggregate shape: (25, 10)


,aspect_type,aspect,n_unique_articles,n_mentions,prob_negative,prob_neutral,prob_positive,sentiment_score,dominant_sentiment,dominant_topic
0,technology,artificial intelligence,84072,84072,0.117409,0.667934,0.214657,0.082073,neutral,Enterprise Technology
1,technology,generative ai,33503,33503,0.121199,0.600127,0.278674,0.143927,neutral,Enterprise Technology
2,technology,chatbot,27612,27612,0.178968,0.620324,0.200708,0.001449,neutral,Enterprise Technology
3,technology,ai model,24618,24618,0.132768,0.600704,0.266527,0.119669,neutral,Enterprise Technology
4,technology,large language model,20954,20954,0.104917,0.673730,0.221353,0.104085,neutral,Enterprise Technology
5,technology,machine learning,19631,19631,0.059871,0.645415,0.294714,0.232184,neutral,Enterprise Technology
6,technology,robotics,6864,6864,0.050148,0.706923,0.242929,0.187063,neutral,Enterprise Technology
7,technology,deep learning,4369,4369,0.041015,0.648089,0.310896,0.270085,neutral,Enterprise Technology
8,technology,natural language processing,4268,4268,0.032746,0.608284,0.358971,0.323336,neutral,Enterprise Technology
9,technology,computer vision,3314,3314,0.029096,0.669795,0.301109,0.278817,neutral,Enterprise Technology


In [80]:
# Aggregate by article
article_agg_dict = {
    "aspect": "size"
}
for c in prob_cols:
    article_agg_dict[c] = "mean"
if "sentiment_score" in absa_df.columns:
    article_agg_dict["sentiment_score"] = "mean"

article_sentiment_df = (
    absa_df.groupby("url")
    .agg(article_agg_dict)
    .rename(columns={"aspect": "n_aspect_mentions"})
    .reset_index()
)

article_dominant_sentiment = (
    absa_df.groupby("url")["sentiment"]
    .agg(safe_mode)
    .reset_index(name="dominant_sentiment")
)

article_dominant_topic = (
    absa_df.groupby("url")["bertopic_topic_label"]
    .agg(safe_mode)
    .reset_index(name="dominant_topic")
)

article_meta = df[["url", "title", "date"]].drop_duplicates(subset=["url"])

article_sentiment_df = article_sentiment_df.merge(article_dominant_sentiment, on="url", how="left")
article_sentiment_df = article_sentiment_df.merge(article_dominant_topic, on="url", how="left")
article_sentiment_df = article_sentiment_df.merge(article_meta, on="url", how="left")

article_sentiment_df = article_sentiment_df.sort_values(
    "n_aspect_mentions",
    ascending=False
).reset_index(drop=True)

print("Article sentiment shape:", article_sentiment_df.shape)
display(article_sentiment_df.head(20))

Article sentiment shape: (134471, 10)


,url,n_aspect_mentions,prob_negative,prob_neutral,prob_positive,sentiment_score,dominant_sentiment,dominant_topic,title,date
0,https://www.techtarget.com/whatis/feature/Arti...,20,0.007558,0.988607,0.003835,0.000000,neutral,Enterprise Technology,Artificial intelligence glossary: 60+ terms to...,2025-08-07
1,https://thenelsonpost.ca/how-research-fuels-ai...,18,0.002007,0.819816,0.178177,0.166667,neutral,Enterprise Technology,How Research Fuels AI Development?,2025-10-17
2,https://venturebeat.com/ai/large-language-mode...,17,0.039188,0.469943,0.490870,0.529412,positive,Enterprise Technology,Large language models broaden AI's reach in in...,2022-12-15
3,https://www.ibm.com/topics/machine-learning-al...,17,0.001882,0.656383,0.341734,0.352941,neutral,Cross-Industry AI News / Outliers,What Is a Machine Learning Algorithm? | IBM,2024-09-02
4,https://www.greenbot.com/ai-systems/,17,0.025968,0.636603,0.337429,0.352941,neutral,Enterprise Technology,"What Is An AI System? Definition, Components &...",2025-02-26
5,https://www.infoworld.com/article/3712283/azur...,17,0.004252,0.973082,0.022666,0.000000,neutral,Enterprise Technology,Azure AI Studio: A nearly complete toolbox for...,2024-01-22
6,https://www.channelasia.tech/article/707067/14...,17,0.005272,0.871249,0.123479,0.117647,neutral,Enterprise Technology,14 popular AI algorithms and their uses - Chan...,2023-05-10
7,https://www.infoworld.com/article/3696970/llms...,16,0.013230,0.948061,0.038709,0.000000,neutral,Enterprise Technology,Large language models and the rise of the AI c...,2023-05-23
8,https://www.ibm.com/think/topics/ai-agent-perc...,16,0.002084,0.600233,0.397683,0.562500,positive,Enterprise Technology,What Is AI Agent Perception? | IBM,2025-03-19
9,https://www.cloudflare.com/en-in/learning/ai/w...,16,0.046371,0.948017,0.005612,-0.062500,neutral,Technology Industry News,What is an LLM (large language model)? | Cloud...,2025-04-24


In [81]:
# Aggregate article sentiment by topic
topic_agg_dict = {
    "url": "nunique",
    "n_aspect_mentions": "sum"
}
for c in prob_cols:
    topic_agg_dict[c] = "mean"
if "sentiment_score" in article_sentiment_df.columns:
    topic_agg_dict["sentiment_score"] = "mean"

topic_sentiment_df = (
    article_sentiment_df.groupby("dominant_topic")
    .agg(topic_agg_dict)
    .rename(columns={
        "url": "n_articles",
        "n_aspect_mentions": "total_aspect_mentions"
    })
    .reset_index()
)

topic_dominant_sentiment = (
    article_sentiment_df.groupby("dominant_topic")["dominant_sentiment"]
    .agg(safe_mode)
    .reset_index(name="dominant_topic_sentiment")
)

topic_sentiment_df = topic_sentiment_df.merge(
    topic_dominant_sentiment,
    on="dominant_topic",
    how="left"
)

topic_sentiment_df = topic_sentiment_df.sort_values(
    "n_articles",
    ascending=False
).reset_index(drop=True)

print("Topic sentiment shape:", topic_sentiment_df.shape)
display(topic_sentiment_df.head(18))

Topic sentiment shape: (18, 8)


,dominant_topic,n_articles,total_aspect_mentions,prob_negative,prob_neutral,prob_positive,sentiment_score,dominant_topic_sentiment
0,Enterprise Technology,60488,272266,0.114487,0.669307,0.216206,0.087240,neutral
1,Cross-Industry AI News / Outliers,35451,157728,0.116044,0.704343,0.179613,0.048681,neutral
2,Consumer & Public Services,6337,30213,0.164874,0.717683,0.117443,-0.066746,neutral
3,Financial Services & Business,5514,23934,0.136029,0.658085,0.205886,0.056795,neutral
4,Corporate AI Announcements,3572,16615,0.018250,0.677359,0.304392,0.282327,neutral
5,Enterprise Software & Cybersecurity,3270,16406,0.011442,0.758684,0.229874,0.215169,neutral
6,Healthcare & Life Sciences,3041,13155,0.056361,0.682187,0.261452,0.199861,neutral
7,Financial Markets & Investing,2808,14211,0.115787,0.568664,0.315548,0.197858,neutral
8,"Media, Journalism & Public Policy",2769,11636,0.134043,0.809334,0.056623,-0.087225,neutral
9,Business Operations & Enterprise Automation,1948,9154,0.014840,0.713867,0.271293,0.256488,neutral


In [83]:
# Top companies by topic
TOP_K = 20

company_by_topic = (
    absa_df[absa_df["aspect_type"] == "company"]
    .groupby(["bertopic_topic_label", "aspect"])
    .agg(
        n_mentions=("aspect", "size"),
        n_unique_articles=("url", "nunique"),
        prob_negative=("prob_negative", "mean"),
        prob_neutral=("prob_neutral", "mean"),
        prob_positive=("prob_positive", "mean"),
        dominant_sentiment=("sentiment", safe_mode)
    )
    .reset_index()
)

# Sort so top mentions appear first inside each topic
company_by_topic = company_by_topic.sort_values(
    ["bertopic_topic_label", "n_mentions"],
    ascending=[True, False]
)

# Dictionary of dataframes: {topic: dataframe}
company_topic_dfs = {
    topic: df.head(TOP_K).reset_index(drop=True)
    for topic, df in company_by_topic.groupby("bertopic_topic_label")
}

# Example: display them
for topic, df_topic in company_topic_dfs.items():
    print(f"\nTopic: {topic}")
    display(df_topic)


Topic: AI Market Commentary


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,AI Market Commentary,My European Quotes,446,446,0.002957,0.991913,0.005130,neutral
1,AI Market Commentary,Motley,269,269,0.142973,0.358172,0.498855,positive
2,AI Market Commentary,Nvidia,261,261,0.105362,0.326768,0.567870,positive
3,AI Market Commentary,Amazon,170,170,0.138835,0.322596,0.538569,positive
4,AI Market Commentary,Microsoft,134,134,0.098655,0.325871,0.575474,positive
5,AI Market Commentary,Alphabet,122,122,0.076321,0.358496,0.565183,positive
6,AI Market Commentary,TradeTalks,116,116,0.002839,0.984264,0.012898,neutral
7,AI Market Commentary,AMD,63,63,0.568562,0.347268,0.084170,negative
8,AI Market Commentary,"Nasdaq, Inc",61,61,0.008065,0.923836,0.068099,neutral
9,AI Market Commentary,Google,60,60,0.149676,0.397481,0.452843,positive



Topic: African Technology Development


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,African Technology Development,Google,176,176,0.089688,0.636120,0.274191,neutral
1,African Technology Development,Arena Holdings,114,114,0.250938,0.586134,0.162927,neutral
2,African Technology Development,Microsoft,98,98,0.230138,0.549366,0.220496,neutral
3,African Technology Development,Facebook,62,62,0.177986,0.612301,0.209713,neutral
4,African Technology Development,Apple,52,52,0.233739,0.569693,0.196568,neutral
5,African Technology Development,Twitter,43,43,0.003685,0.971491,0.024824,neutral
6,African Technology Development,Meta,41,41,0.239212,0.504206,0.256582,neutral
7,African Technology Development,Instagram,40,40,0.145825,0.620719,0.233456,neutral
8,African Technology Development,Amazon,39,39,0.204178,0.518745,0.277076,neutral
9,African Technology Development,Artificial Intelligence,38,38,0.061675,0.741466,0.196859,neutral



Topic: Business Operations & Enterprise Automation


...

,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Publishing & Intellectual Property,Google,303,303,0.169353,0.639666,0.190981,neutral
1,Publishing & Intellectual Property,Cox Media Group,269,269,0.011635,0.981099,0.007265,neutral
2,Publishing & Intellectual Property,Cox Media Group Television,263,263,0.047535,0.937098,0.015368,neutral
3,Publishing & Intellectual Property,Microsoft,210,210,0.227568,0.575037,0.197395,neutral
4,Publishing & Intellectual Property,Amazon,152,152,0.265035,0.496873,0.238092,neutral
5,Publishing & Intellectual Property,Facebook,88,88,0.057466,0.891584,0.050951,neutral
6,Publishing & Intellectual Property,Instagram,75,75,0.156068,0.792501,0.051431,neutral
7,Publishing & Intellectual Property,Meta,75,75,0.233879,0.674793,0.091328,neutral
8,Publishing & Intellectual Property,Cox Media Group(Opens,72,72,0.030501,0.921832,0.047667,neutral
9,Publishing & Intellectual Property,Apple,64,64,0.121109,0.633679,0.245213,neutral



Topic: Technology Industry News


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Technology Industry News,Google,322,322,0.223937,0.414761,0.361301,neutral
1,Technology Industry News,Microsoft,172,172,0.266410,0.400980,0.332610,neutral
2,Technology Industry News,Amazon,133,133,0.237846,0.340833,0.421321,positive
3,Technology Industry News,Meta,87,87,0.231492,0.444552,0.323956,neutral
4,Technology Industry News,Nvidia,78,78,0.195453,0.362974,0.441572,positive
5,Technology Industry News,Apple,72,72,0.194503,0.426804,0.378693,neutral
6,Technology Industry News,Instagram,29,29,0.271810,0.434285,0.293904,neutral
7,Technology Industry News,AMD,28,28,0.310589,0.234084,0.455327,positive
8,Technology Industry News,Intel,27,27,0.243651,0.297578,0.458771,positive
9,Technology Industry News,Meta Platforms Inc,27,27,0.410295,0.207078,0.382627,positive



Topic: Workforce & Compensation


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Workforce & Compensation,Google,973,973,0.003874,0.990660,0.005466,neutral
1,Workforce & Compensation,Location,822,822,0.002911,0.991805,0.005283,neutral
2,Workforce & Compensation,Data Scientist,630,630,0.003208,0.992167,0.004626,neutral
3,Workforce & Compensation,Levels Fyi Inc.ChangeDownload,300,300,0.004183,0.969771,0.026046,neutral
4,Workforce & Compensation,Levels.fyi,276,276,0.002913,0.992191,0.004896,neutral
5,Workforce & Compensation,FAANG,150,150,0.007367,0.950661,0.041972,neutral
6,Workforce & Compensation,ExperienceTotal,97,97,0.002560,0.987344,0.010096,neutral
7,Workforce & Compensation,Heatmap,82,82,0.003353,0.989988,0.006659,neutral
8,Workforce & Compensation,LinkedIn,52,52,0.011191,0.982386,0.006424,neutral
9,Workforce & Compensation,Data Science,30,30,0.003038,0.991096,0.005866,neutral


In [84]:
# Top technologies by topic
technology_by_topic = (
    absa_df[absa_df["aspect_type"] == "technology"]
    .groupby(["bertopic_topic_label", "aspect"])
    .agg(
        n_mentions=("aspect", "size"),
        n_unique_articles=("url", "nunique"),
        prob_negative=("prob_negative", "mean"),
        prob_neutral=("prob_neutral", "mean"),
        prob_positive=("prob_positive", "mean"),
        dominant_sentiment=("sentiment", safe_mode)
    )
    .reset_index()
)

# Sort so the most-mentioned technologies appear first within each topic
technology_by_topic = technology_by_topic.sort_values(
    ["bertopic_topic_label", "n_mentions"],
    ascending=[True, False]
)

# Dictionary of dataframes: {topic: dataframe}
technology_topic_dfs = {
    topic: df.head(TOP_K).reset_index(drop=True)
    for topic, df in technology_by_topic.groupby("bertopic_topic_label")
}

# Display results
for topic, df_topic in technology_topic_dfs.items():
    print(f"\nTopic: {topic}")
    display(df_topic)


Topic: AI Market Commentary


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,AI Market Commentary,artificial intelligence,664,664,0.092807,0.445029,0.462164,positive
1,AI Market Commentary,generative ai,187,187,0.055935,0.282632,0.661433,positive
2,AI Market Commentary,ai model,168,168,0.071579,0.305471,0.622949,positive
3,AI Market Commentary,large language model,123,123,0.045672,0.293216,0.661112,positive
4,AI Market Commentary,machine learning,88,88,0.007129,0.259880,0.732991,positive
5,AI Market Commentary,chatbot,63,63,0.099775,0.480155,0.420070,neutral
6,AI Market Commentary,robotics,58,58,0.118923,0.307892,0.573185,positive
7,AI Market Commentary,autonomous driving,21,21,0.087651,0.273008,0.639341,positive
8,AI Market Commentary,foundation model,15,15,0.002401,0.374357,0.623241,positive
9,AI Market Commentary,computer vision,12,12,0.019746,0.422392,0.557862,positive



Topic: African Technology Development


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,African Technology Development,artificial intelligence,782,782,0.092681,0.649772,0.257547,neutral
1,African Technology Development,machine learning,228,228,0.064592,0.747435,0.187973,neutral
2,African Technology Development,generative ai,175,175,0.114367,0.593330,0.292303,neutral
3,African Technology Development,chatbot,168,168,0.145653,0.545185,0.309162,neutral
4,African Technology Development,ai model,154,154,0.149050,0.573921,0.277030,neutral
5,African Technology Development,large language model,108,108,0.106003,0.691588,0.202409,neutral
6,African Technology Development,robotics,68,68,0.081263,0.540423,0.378314,neutral
7,African Technology Development,deep learning,31,31,0.091560,0.655554,0.252887,neutral
8,African Technology Development,natural language processing,31,31,0.140363,0.580468,0.279169,neutral
9,African Technology Development,computer vision,23,23,0.086364,0.481865,0.431771,positive



Topic: Business Operations & Enterprise Automation


...

,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Publishing & Intellectual Property,artificial intelligence,1006,1006,0.207513,0.638376,0.154111,neutral
1,Publishing & Intellectual Property,chatbot,390,390,0.269421,0.588842,0.141736,neutral
2,Publishing & Intellectual Property,generative ai,345,345,0.207578,0.621244,0.171178,neutral
3,Publishing & Intellectual Property,ai model,196,196,0.255186,0.615162,0.129653,neutral
4,Publishing & Intellectual Property,large language model,151,151,0.214903,0.705518,0.079578,neutral
5,Publishing & Intellectual Property,machine learning,58,58,0.091198,0.709284,0.199518,neutral
6,Publishing & Intellectual Property,foundation model,35,35,0.272902,0.699320,0.027778,neutral
7,Publishing & Intellectual Property,computer vision,18,18,0.020231,0.746323,0.233446,neutral
8,Publishing & Intellectual Property,robotics,17,17,0.237788,0.654322,0.107891,neutral
9,Publishing & Intellectual Property,neural network,12,12,0.046473,0.885005,0.068523,neutral



Topic: Technology Industry News


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Technology Industry News,artificial intelligence,922,922,0.236713,0.344931,0.418356,positive
1,Technology Industry News,generative ai,384,384,0.166299,0.399129,0.434572,positive
2,Technology Industry News,ai model,326,326,0.197003,0.410244,0.392752,neutral
3,Technology Industry News,machine learning,309,309,0.129159,0.509292,0.361549,neutral
4,Technology Industry News,large language model,214,214,0.168749,0.521958,0.309293,neutral
5,Technology Industry News,chatbot,171,171,0.265144,0.461854,0.273002,neutral
6,Technology Industry News,robotics,61,61,0.109708,0.483778,0.406513,neutral
7,Technology Industry News,natural language processing,58,58,0.140434,0.499126,0.360440,neutral
8,Technology Industry News,multimodal ai,38,38,0.079630,0.333379,0.586991,positive
9,Technology Industry News,neural network,34,34,0.031167,0.618108,0.350725,neutral



Topic: Workforce & Compensation


,bertopic_topic_label,aspect,n_mentions,n_unique_articles,prob_negative,prob_neutral,prob_positive,dominant_sentiment
0,Workforce & Compensation,machine learning,29,29,0.002876,0.963114,0.034010,neutral
1,Workforce & Compensation,large language model,21,21,0.003436,0.990246,0.006318,neutral
2,Workforce & Compensation,robotics,3,3,0.002439,0.990912,0.006649,neutral
3,Workforce & Compensation,artificial intelligence,2,2,0.001767,0.603225,0.395009,neutral
4,Workforce & Compensation,ai model,1,1,0.006433,0.902655,0.090912,neutral
5,Workforce & Compensation,computer vision,1,1,0.002493,0.978031,0.019476,neutral
6,Workforce & Compensation,generative ai,1,1,0.001458,0.988154,0.010389,neutral
7,Workforce & Compensation,reinforcement learning,1,1,0.001325,0.934061,0.064614,neutral
8,Workforce & Compensation,supervised learning,1,1,0.001747,0.045404,0.952849,positive


In [85]:
# File paths
# Main file paths
absa_detail_path = os.path.join(OUTPUT_DIR, "absa_detailed_aspects.parquet")
aspect_agg_path = os.path.join(OUTPUT_DIR, "absa_aspect_aggregate.parquet")
company_aspect_path = os.path.join(OUTPUT_DIR, "absa_company_aspects.parquet")
technology_aspect_path = os.path.join(OUTPUT_DIR, "absa_technology_aspects.parquet")
article_sent_path = os.path.join(OUTPUT_DIR, "article_sentiment.parquet")
topic_sent_path = os.path.join(OUTPUT_DIR, "topic_sentiment.parquet")
company_by_topic_path = os.path.join(OUTPUT_DIR, "company_by_topic.parquet")
technology_by_topic_path = os.path.join(OUTPUT_DIR, "technology_by_topic.parquet")

# Save main tables
absa_df.to_parquet(absa_detail_path, index=False)

aspect_agg_df.to_parquet(aspect_agg_path, index=False)
company_aspect_agg_df.to_parquet(company_aspect_path, index=False)
technology_aspect_agg_df.to_parquet(technology_aspect_path, index=False)

article_sentiment_df.to_parquet(article_sent_path, index=False)
topic_sentiment_df.to_parquet(topic_sent_path, index=False)

# Save full by-topic tables
company_by_topic.to_parquet(company_by_topic_path, index=False)
technology_by_topic.to_parquet(technology_by_topic_path, index=False)

# Save each topic-specific company table
company_topic_dir = os.path.join(OUTPUT_DIR, "top_companies_by_topic")
os.makedirs(company_topic_dir, exist_ok=True)

for topic, df_topic in company_topic_dfs.items():
    topic_slug = re.sub(r"[^\w\s-]", "", str(topic).lower())
    topic_slug = re.sub(r"\s+", "_", topic_slug.strip())
    topic_slug = re.sub(r"_+", "_", topic_slug)

    path = os.path.join(company_topic_dir, f"{topic_slug}.parquet")
    df_topic.to_parquet(path, index=False)

# Save each topic-specific technology table
technology_topic_dir = os.path.join(OUTPUT_DIR, "top_technologies_by_topic")
os.makedirs(technology_topic_dir, exist_ok=True)

for topic, df_topic in technology_topic_dfs.items():
    topic_slug = re.sub(r"[^\w\s-]", "", str(topic).lower())
    topic_slug = re.sub(r"\s+", "_", topic_slug.strip())
    topic_slug = re.sub(r"_+", "_", topic_slug)

    path = os.path.join(technology_topic_dir, f"{topic_slug}.parquet")
    df_topic.to_parquet(path, index=False)

print("\nSaved files:")
print(absa_detail_path)
print(aspect_agg_path)
print(company_aspect_path)
print(technology_aspect_path)
print(article_sent_path)
print(topic_sent_path)
print(company_by_topic_path)
print(technology_by_topic_path)
print(company_topic_dir)
print(technology_topic_dir)


Saved files:
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/absa_detailed_aspects.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/absa_aspect_aggregate.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/absa_company_aspects.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/absa_technology_aspects.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/article_sentiment.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/topic_sentiment.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/company_by_topic.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/technology_by_topic.parquet
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/top_companies_by_topic
/content/drive/MyDrive/UChicago/Maste

In [86]:
# Graphs
sentiment_order = ["negative", "neutral", "positive"]

def plot_grouped_sentiment_bars(
    counts_df,
    category_col,
    title,
    save_path,
    figsize=(12, 6),
    rotate_xticks=45
):
    if counts_df.empty:
        return

    pivot_df = (
        counts_df.pivot(index=category_col, columns="sentiment", values="count")
        .fillna(0)
        .astype(int)
    )

    pivot_df = ensure_sentiment_columns(pivot_df, sentiments=sentiment_order)
    pivot_df = pivot_df.reset_index()

    labels = wrap_or_shorten_labels(pivot_df[category_col].tolist(), max_len=35)
    x = np.arange(len(labels))
    width = 0.25

    plt.figure(figsize=figsize)
    plt.bar(x - width, pivot_df["negative"], width=width, label="Negative")
    plt.bar(x,         pivot_df["neutral"],  width=width, label="Neutral")
    plt.bar(x + width, pivot_df["positive"], width=width, label="Positive")

    plt.xticks(x, labels, rotation=rotate_xticks, ha="right")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_single_topic_article_sentiment(topic_display, topic_article_counts, save_path):
    vals = [int(topic_article_counts.get(s, 0)) for s in sentiment_order]

    plt.figure(figsize=(8, 5))
    plt.bar(["Negative", "Neutral", "Positive"], vals)
    plt.ylabel("# of Articles")
    plt.title(f"Article Sentiment Counts\n{shorten(str(topic_display), width=90, placeholder='...')}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_sentiment_proportion_over_time(prop_df, topic_display, save_path, freq_label="Month"):
    if prop_df.empty:
        return

    plt.figure(figsize=(12, 6))
    plt.plot(prop_df["time_period"], prop_df["negative"], marker="o", label="Negative")
    plt.plot(prop_df["time_period"], prop_df["neutral"], marker="o", label="Neutral")
    plt.plot(prop_df["time_period"], prop_df["positive"], marker="o", label="Positive")

    plt.ylim(0, 1)
    plt.ylabel("Proportion of Articles")
    plt.xlabel(freq_label)
    plt.title(f"Sentiment Proportions Over Time\n{shorten(str(topic_display), width=90, placeholder='...')}")
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

In [87]:
# Build topic num label matching
topic_meta = (
    bertopic_small[["bertopic_topic_text", "bertopic_topic_label"]]
    .drop_duplicates()
    .rename(columns={"bertopic_topic_text": "topic_num"})
    .copy()
)

topic_num_map = dict(zip(topic_meta["bertopic_topic_label"], topic_meta["topic_num"]))

In [88]:
# Build graph source tables
# A) Article-level sentiment counts by topic
article_sentiment_counts_by_topic = (
    article_sentiment_df.groupby(["dominant_topic", "dominant_sentiment"])
    .size()
    .reset_index(name="count")
)
article_sentiment_counts_by_topic.to_parquet(
    os.path.join(OUTPUT_DIR, "article_topic_sentiment_counts.parquet"),
    index=False
)

# B) Company sentiment counts by topic + company
topic_company_sentiment_counts = (
    absa_df[absa_df["aspect_type"] == "company"]
    .groupby(["bertopic_topic_label", "aspect", "sentiment"])
    .size()
    .reset_index(name="count")
)
topic_company_sentiment_counts.to_parquet(
    os.path.join(OUTPUT_DIR, "topic_company_sentiment_counts.parquet"),
    index=False
)

# C) Technology sentiment counts by topic + technology
topic_technology_sentiment_counts = (
    absa_df[absa_df["aspect_type"] == "technology"]
    .groupby(["bertopic_topic_label", "aspect", "sentiment"])
    .size()
    .reset_index(name="count")
)
topic_technology_sentiment_counts.to_parquet(
    os.path.join(OUTPUT_DIR, "topic_technology_sentiment_counts.parquet"),
    index=False
)

# D) Top 10 companies within each topic
topic_top10_companies = (
    absa_df[absa_df["aspect_type"] == "company"]
    .groupby(["bertopic_topic_label", "aspect"])
    .size()
    .reset_index(name="n_mentions")
    .sort_values(["bertopic_topic_label", "n_mentions"], ascending=[True, False])
    .groupby("bertopic_topic_label", group_keys=False)
    .head(10)
    .copy()
)
topic_top10_companies.to_parquet(
    os.path.join(OUTPUT_DIR, "topic_top10_companies.parquet"),
    index=False
)

# E) Top 10 technologies within each topic
topic_top10_technologies = (
    absa_df[absa_df["aspect_type"] == "technology"]
    .groupby(["bertopic_topic_label", "aspect"])
    .size()
    .reset_index(name="n_mentions")
    .sort_values(["bertopic_topic_label", "n_mentions"], ascending=[True, False])
    .groupby("bertopic_topic_label", group_keys=False)
    .head(10)
    .copy()
)
topic_top10_technologies.to_parquet(
    os.path.join(OUTPUT_DIR, "topic_top10_technologies.parquet"),
    index=False
)

In [89]:
# Time series sentiment proportions
article_sentiment_time_df = article_sentiment_df.copy()
article_sentiment_time_df["date"] = pd.to_datetime(article_sentiment_time_df["date"], errors="coerce")
article_sentiment_time_df = article_sentiment_time_df[
    article_sentiment_time_df["date"].notna()
].copy()

if len(article_sentiment_time_df) > 0:
    TIME_FREQ = "M"

    article_sentiment_time_df["time_period"] = (
        article_sentiment_time_df["date"]
        .dt.to_period(TIME_FREQ)
        .dt.to_timestamp()
    )

    topic_time_sentiment_counts = (
        article_sentiment_time_df.groupby(
            ["dominant_topic", "time_period", "dominant_sentiment"]
        )
        .size()
        .reset_index(name="count")
    )

    topic_time_sentiment_pivot = (
        topic_time_sentiment_counts
        .pivot_table(
            index=["dominant_topic", "time_period"],
            columns="dominant_sentiment",
            values="count",
            fill_value=0
        )
        .reset_index()
    )

    topic_time_sentiment_pivot = ensure_sentiment_columns(
        topic_time_sentiment_pivot.set_index(["dominant_topic", "time_period"]),
        sentiments=sentiment_order
    ).reset_index()

    topic_time_sentiment_pivot["total_articles"] = (
        topic_time_sentiment_pivot["negative"] +
        topic_time_sentiment_pivot["neutral"] +
        topic_time_sentiment_pivot["positive"]
    )

    for s in sentiment_order:
        topic_time_sentiment_pivot[f"{s}_prop"] = np.where(
            topic_time_sentiment_pivot["total_articles"] > 0,
            topic_time_sentiment_pivot[s] / topic_time_sentiment_pivot["total_articles"],
            0.0
        )

    topic_time_sentiment_pivot.to_parquet(
        os.path.join(OUTPUT_DIR, "topic_time_sentiment_proportions.parquet"),
        index=False
    )
else:
    topic_time_sentiment_pivot = pd.DataFrame()
    print("No valid date column found. Skipping time graphs.")

In [93]:
# Create graph files
all_topics = sorted(article_sentiment_df["dominant_topic"].dropna().astype(str).unique())

saved_topic_article_graphs = []
saved_topic_company_graphs = []
saved_topic_technology_graphs = []
saved_topic_time_graphs = []

for topic_label in tqdm(all_topics, total=len(all_topics), desc="Saving topic graphs"):
    topic_num = topic_num_map.get(topic_label, np.nan)

    if pd.notna(topic_num):
        topic_num_int = int(topic_num)
        topic_num_slug = f"topic_{topic_num_int}"
        topic_display = f"Topic {topic_num_int}: {topic_label}"
    else:
        topic_num_slug = "topic_unknown"
        topic_display = f"Topic Unknown: {topic_label}"

    topic_label_slug = slugify(topic_label)

    # ------------------------------------------
    # 1) Article sentiment counts for this topic
    # ------------------------------------------
    topic_article_counts = (
        article_sentiment_df.loc[
            article_sentiment_df["dominant_topic"] == topic_label,
            "dominant_sentiment"
        ]
        .value_counts()
    )

    topic_article_plot_path = os.path.join(
        TOPIC_GRAPH_DIR,
        f"{topic_num_slug}_{topic_label_slug}_article_sentiment_counts.png"
    )

    plot_single_topic_article_sentiment(
        topic_display=topic_display,
        topic_article_counts=topic_article_counts,
        save_path=topic_article_plot_path
    )
    saved_topic_article_graphs.append(topic_article_plot_path)

    # ------------------------------------------
    # 2) Top 10 companies sentiment counts
    # ------------------------------------------
    top_companies_this_topic = topic_top10_companies.loc[
        topic_top10_companies["bertopic_topic_label"] == topic_label,
        "aspect"
    ].tolist()

    if len(top_companies_this_topic) > 0:
        company_counts_this_topic = topic_company_sentiment_counts[
            (topic_company_sentiment_counts["bertopic_topic_label"] == topic_label) &
            (topic_company_sentiment_counts["aspect"].isin(top_companies_this_topic))
        ].copy()

        company_order = (
            topic_top10_companies.loc[
                topic_top10_companies["bertopic_topic_label"] == topic_label,
                ["aspect", "n_mentions"]
            ]
            .sort_values("n_mentions", ascending=False)["aspect"]
            .tolist()
        )

        company_counts_this_topic["aspect"] = pd.Categorical(
            company_counts_this_topic["aspect"],
            categories=company_order,
            ordered=True
        )
        company_counts_this_topic = company_counts_this_topic.sort_values("aspect")

        topic_company_plot_path = os.path.join(
            COMPANY_GRAPH_DIR,
            f"{topic_num_slug}_{topic_label_slug}_top10_companies_sentiment_counts.png"
        )

        plot_grouped_sentiment_bars(
            counts_df=company_counts_this_topic.rename(columns={"aspect": "company"}),
            category_col="company",
            title=f"Top 10 Companies by Sentiment Count\n{shorten(str(topic_display), width=90, placeholder='...')}",
            save_path=topic_company_plot_path,
            figsize=(14, 6),
            rotate_xticks=45
        )
        saved_topic_company_graphs.append(topic_company_plot_path)

    # ------------------------------------------
    # 3) Top 10 technologies sentiment counts
    # ------------------------------------------
    top_tech_this_topic = topic_top10_technologies.loc[
        topic_top10_technologies["bertopic_topic_label"] == topic_label,
        "aspect"
    ].tolist()

    if len(top_tech_this_topic) > 0:
        tech_counts_this_topic = topic_technology_sentiment_counts[
            (topic_technology_sentiment_counts["bertopic_topic_label"] == topic_label) &
            (topic_technology_sentiment_counts["aspect"].isin(top_tech_this_topic))
        ].copy()

        tech_order = (
            topic_top10_technologies.loc[
                topic_top10_technologies["bertopic_topic_label"] == topic_label,
                ["aspect", "n_mentions"]
            ]
            .sort_values("n_mentions", ascending=False)["aspect"]
            .tolist()
        )

        tech_counts_this_topic["aspect"] = pd.Categorical(
            tech_counts_this_topic["aspect"],
            categories=tech_order,
            ordered=True
        )
        tech_counts_this_topic = tech_counts_this_topic.sort_values("aspect")

        topic_tech_plot_path = os.path.join(
            TECH_GRAPH_DIR,
            f"{topic_num_slug}_{topic_label_slug}_top10_technologies_sentiment_counts.png"
        )

        plot_grouped_sentiment_bars(
            counts_df=tech_counts_this_topic.rename(columns={"aspect": "technology"}),
            category_col="technology",
            title=f"Top 10 Technologies by Sentiment Count\n{shorten(str(topic_display), width=90, placeholder='...')}",
            save_path=topic_tech_plot_path,
            figsize=(14, 6),
            rotate_xticks=45
        )
        saved_topic_technology_graphs.append(topic_tech_plot_path)

    # ------------------------------------------
    # 4) Sentiment proportions over time
    # ------------------------------------------
    if not topic_time_sentiment_pivot.empty:
        topic_time_df = topic_time_sentiment_pivot[
            topic_time_sentiment_pivot["dominant_topic"] == topic_label
        ].copy()

        if not topic_time_df.empty:
            topic_time_df = topic_time_df.sort_values("time_period")

            prop_df = topic_time_df[[
                "time_period", "negative_prop", "neutral_prop", "positive_prop"
            ]].rename(columns={
                "negative_prop": "negative",
                "neutral_prop": "neutral",
                "positive_prop": "positive"
            })

            topic_time_plot_path = os.path.join(
                TIME_GRAPH_DIR,
                f"{topic_num_slug}_{topic_label_slug}_sentiment_proportions_over_time.png"
            )

            plot_sentiment_proportion_over_time(
                prop_df=prop_df,
                topic_display=topic_display,
                save_path=topic_time_plot_path,
                freq_label="Month"
            )
            saved_topic_time_graphs.append(topic_time_plot_path)

Saving topic graphs:   0%|          | 0/18 [00:00<?, ?it/s]

In [94]:
# Graph saves
graph_manifest = pd.DataFrame({
    "graph_type": (
        ["topic_article_sentiment"] * len(saved_topic_article_graphs) +
        ["topic_top10_companies_sentiment"] * len(saved_topic_company_graphs) +
        ["topic_top10_technologies_sentiment"] * len(saved_topic_technology_graphs) +
        ["topic_sentiment_proportions_over_time"] * len(saved_topic_time_graphs)
    ),
    "file_path": (
        saved_topic_article_graphs +
        saved_topic_company_graphs +
        saved_topic_technology_graphs +
        saved_topic_time_graphs
    )
})

graph_manifest_path = os.path.join(OUTPUT_DIR, "graph_manifest.parquet")
graph_manifest.to_parquet(graph_manifest_path, index=False)

print("\nGraph folders:")
print("Topic article sentiment graphs:", TOPIC_GRAPH_DIR)
print("Topic top-10 company graphs:", COMPANY_GRAPH_DIR)
print("Topic top-10 technology graphs:", TECH_GRAPH_DIR)
print("Topic time-series sentiment graphs:", TIME_GRAPH_DIR)

print("\nSaved graph manifest:")
print(graph_manifest_path)

print("\nNumber of graphs saved:")
print("Article sentiment by topic:", len(saved_topic_article_graphs))
print("Top-10 companies by topic:", len(saved_topic_company_graphs))
print("Top-10 technologies by topic:", len(saved_topic_technology_graphs))
print("Sentiment proportions over time:", len(saved_topic_time_graphs))

display(graph_manifest.head(20))


Graph folders:
Topic article sentiment graphs: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/graphs/topic_article_sentiment
Topic top-10 company graphs: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/graphs/topic_top10_companies_sentiment
Topic top-10 technology graphs: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/graphs/topic_top10_technologies_sentiment
Topic time-series sentiment graphs: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/graphs/topic_sentiment_over_time

Saved graph manifest:
/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/absa_outputs/graph_manifest.parquet

Number of graphs saved:
Article sentiment by topic: 18
Top-10 companies by topic: 18
Top-10 technologies by topic: 18
Sentiment proportions over time: 18


,graph_type,file_path
0,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
1,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
2,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
3,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
4,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
5,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
6,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
7,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
8,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
9,topic_article_sentiment,/content/drive/MyDrive/UChicago/Masters/Winter...
